In [ ]:
models_order = comparison_df.index.tolist()
metrics_4    = ["accuracy", "precision", "recall", "f1"]
colors_4     = sns.color_palette("Set2", 4)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# ── 1. Grouped bar: all 4 metrics × 6 models (THE MONEY CHART) ───────────────
ax = axes[0, 0]
x  = np.arange(len(models_order))
w  = 0.18
for i, (metric, color) in enumerate(zip(metrics_4, colors_4)):
    vals = [comparison_df.loc[m, metric] for m in models_order]
    bars = ax.bar(x + i*w, vals, width=w, label=metric.title(),
                  color=color, edgecolor="white")
ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(models_order, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("All 4 Metrics × 6 Models", fontweight="bold")
ax.legend(loc="lower right")
ax.axhline(0.73, color="gray", linestyle="--", linewidth=0.8,
           label="Dummy accuracy (73%)")

# ── 2. ROC-AUC bar chart ─────────────────────────────────────────────────────
ax = axes[0, 1]
roc_vals = [comparison_df.loc[m, "roc_auc"] for m in models_order]
colors_roc = sns.color_palette("viridis", len(models_order))
bars = ax.bar(models_order, roc_vals, color=colors_roc, edgecolor="white")
for bar, val in zip(bars, roc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=9)
ax.set_xticklabels(models_order, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.7, 1.0)
ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="Random (0.5)")
ax.set_title("ROC-AUC Scores", fontweight="bold")
ax.legend()

# ── 3. Training time ─────────────────────────────────────────────────────────
ax = axes[1, 0]
times = [comparison_df.loc[m, "train_time_seconds"] for m in models_order]
bars  = ax.barh(models_order[::-1], times[::-1],
                color=sns.color_palette("Set2", len(models_order)), edgecolor="white")
for bar, val in zip(bars, times[::-1]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}s", va="center", fontsize=9)
ax.set_xlabel("Training Time (seconds)")
ax.set_title("Training Time Comparison", fontweight="bold")

# ── 4. F1 vs Training Time scatter ───────────────────────────────────────────
ax = axes[1, 1]
scatter_colors = sns.color_palette("tab10", len(models_order))
for i, m in enumerate(models_order):
    ax.scatter(comparison_df.loc[m, "train_time_seconds"],
               comparison_df.loc[m, "f1"],
               s=150, color=scatter_colors[i], zorder=3, label=m)
    ax.annotate(m, xy=(comparison_df.loc[m, "train_time_seconds"],
                       comparison_df.loc[m, "f1"]),
                xytext=(5, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Training Time (seconds)")
ax.set_ylabel("F1 Score")
ax.set_title("Efficiency: F1 vs Training Time\n(top-right = best)", fontweight="bold")
ax.legend(fontsize=7, loc="lower right")

plt.suptitle("Model Comparison Dashboard", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_11_model_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

---
## Section 3 — Comparison Visualisations

In [ ]:
trainer = ModelTrainer(X_train_proc, y_train, X_test_proc, y_test)
comparison_df = trainer.train_all()

print("\n\n" + "="*55)
print("  FINAL COMPARISON (sorted by F1)")
print("="*55)
display(comparison_df.style
    .background_gradient(subset=["f1","roc_auc","recall"], cmap="YlGn")
    .format({c: "{:.4f}" for c in comparison_df.columns if comparison_df[c].dtype == float})
    .set_caption("Model Comparison — All Metrics")
)

---
## Section 2 — Train All 6 Models

We train: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, SVM.

Each uses `class_weight='balanced'` (or `scale_pos_weight` for XGBoost) to handle the 73/27 imbalance.

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = engineer_features(clean_data(load_data(RAW_PATH)))
X_train, X_test, y_train, y_test = split_data(df)

# Preprocessor
numerical_cols   = ["tenure","MonthlyCharges","TotalCharges",
                    "monthly_tenure_ratio","total_services","avg_monthly_spend","contract_risk"]
categorical_cols = ["gender","SeniorCitizen","Partner","Dependents","PhoneService",
                    "MultipleLines","InternetService","OnlineSecurity","OnlineBackup",
                    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies",
                    "Contract","PaperlessBilling","PaymentMethod","tenure_group"]

preprocessor = joblib.load("../models/preprocessor.joblib")
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"Train: {X_train_proc.shape}  |  Churn rate: {y_train.mean():.1%}")
print(f"Test:  {X_test_proc.shape}   |  Churn rate: {y_test.mean():.1%}")

---
## Section 1 — Load & Prepare Data

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from src.preprocessing import load_data, clean_data, split_data
from src.features      import engineer_features, create_preprocessor
from src.train         import ModelTrainer

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})
print("✅ Imports OK")

# 03 — Model Training
## ML Classification & Prediction System | Telco Customer Churn

**Goal:** Train 6 different classification algorithms and compare them rigorously.

> **Why train multiple models?**  
> No single model is best for every problem. By training several, we learn the performance landscape and can make an informed choice for tuning.

**Sections:**
1. Load processed data
2. Train all 6 models
3. Comparison visualisations (the money chart)
4. Analysis — why did each model perform the way it did?
5. Classification reports for top 3 models
6. Select top 2 models for hyperparameter tuning